# 01c — Feasibility and scale

**Task:** M1-T03 (feasibility and scale justification).

**Purpose:** show that the chosen AOIs yield far more than the Milestone 1 minimum of 10,000 records with 15+ attributes, using the real AOI squares, and project the Sentinel Hub processing-unit (PU) cost of the planned extraction against the budget.

**Inputs:** `config/aois.geojson`, `config/project.yaml`, `reports/results/m1_aoi_availability.csv` (acquisition dates per AOI-year from M1-T02).

**Outputs:** `reports/results/m1_feasibility.csv`, `m1_attributes.csv`, `m1_pu_projection.csv`.

**Run time:** seconds. No network access.

In [1]:
%load_ext autoreload
%autoreload 2

import json
import platform

import numpy as np
import pandas as pd

import desertification
from desertification.config import get_path, load_config
from desertification.feasibility import complete_seasons, composite_pu, grid_shape, season_fraction
from desertification.schema import CELL_SEASON_KEYS, attribute_names, cell_season_columns

cfg = load_config()
SEED = cfg["project"]["seed"]
rng = np.random.default_rng(SEED)
RESULTS = get_path(cfg, "results")
print(f"Python {platform.python_version()} | desertification {desertification.__version__} | pandas {pd.__version__}")

Python 3.12.3 | desertification 0.1.0 | pandas 3.1.1


## 1. Records per AOI

Each AOI is a 20 km UTM square divided into 100 m cells (`grid.cell_size_m`), i.e. 5 × 5 pixels at 20 m. The main Milestone 1 table, `cell_season`, has one row per cell, year and season (ROADMAP §7.2). Only **complete** seasons inside the 2019–2025 study period are counted; a season that crosses the year end (e.g. Aral's December–March wet season) gives 6 rather than 7.

In [2]:
aois = json.loads(get_path(cfg, "aois").read_text())["features"]
study = cfg["periods"]["study"]
cell_m, px_m = cfg["grid"]["cell_size_m"], cfg["grid"]["pixel_resolution_m"]
n_months = (int(study["end"][:4]) - int(study["start"][:4]) + 1) * 12

rows = []
for f in aois:
    p = f["properties"]
    nx, ny = grid_shape(p["bbox_utm"], cell_m)
    px, py = grid_shape(p["bbox_utm"], px_m)
    wet = complete_seasons(p["wet_season_months"], study["start"], study["end"])
    dry = complete_seasons(p["dry_season_months"], study["start"], study["end"])
    n_cells = nx * ny
    rows.append({
        "aoi_id": p["aoi_id"], "cells": n_cells, "pixels_20m": px * py,
        "wet_seasons": len(wet), "dry_seasons": len(dry),
        "cell_season_records": n_cells * (len(wet) + len(dry)),
        "cell_monthly_records": n_cells * n_months,
    })
feasibility = pd.DataFrame(rows)
total = feasibility.drop(columns="aoi_id").sum().to_dict() | {"aoi_id": "TOTAL"}
feasibility = pd.concat([feasibility, pd.DataFrame([total])], ignore_index=True)
feasibility.to_csv(RESULTS / "m1_feasibility.csv", index=False)
feasibility

,aoi_id,cells,pixels_20m,wet_seasons,dry_seasons,cell_season_records,cell_monthly_records
0,aral,40000,1000000,6,7,520000,3360000
1,ferlo,40000,1000000,7,7,560000,3360000
2,fakara,40000,1000000,7,6,520000,3360000
3,kubuqi,40000,1000000,7,6,520000,3360000
4,urmia,40000,1000000,7,7,560000,3360000
5,erg,40000,1000000,7,7,560000,3360000
6,TOTAL,240000,6000000,41,40,3240000,20160000


## 2. Attributes

The `cell_season` columns come from `desertification.schema`, which a unit test checks against ROADMAP §7.2. Key columns (`cell_id`, `aoi_id`) are not counted as attributes. The within-cell standard deviations are optional, so both counts are shown.

In [3]:
columns = cell_season_columns(cfg["sentinel2"]["bands"], include_band_std=True)
attributes = pd.DataFrame(columns, columns=["column", "dtype", "description"])
attributes["is_key"] = attributes["column"].isin(CELL_SEASON_KEYS)
attributes.to_csv(RESULTS / "m1_attributes.csv", index=False)
n_attr_std = len(attribute_names(columns, CELL_SEASON_KEYS))
n_attr = len(attribute_names(cell_season_columns(cfg["sentinel2"]["bands"], include_band_std=False), CELL_SEASON_KEYS))
records = int(feasibility.loc[feasibility.aoi_id == "TOTAL", "cell_season_records"].item())
smallest = int(feasibility.loc[feasibility.aoi_id != "TOTAL", "cell_season_records"].min())
print(f"cell_season: {records:,} records in total (smallest AOI alone: {smallest:,}); "
      f"{n_attr_std} attributes with band std, {n_attr} without")
print("Milestone 1 minimum: 10,000 records and 15 attributes ->",
      "MET" if smallest >= 10_000 and n_attr >= 15 else "NOT MET")
attributes

cell_season: 3,240,000 records in total (smallest AOI alone: 520,000); 34 attributes with band std, 24 without
Milestone 1 minimum: 10,000 records and 15 attributes -> MET


,column,dtype,description,is_key
0,cell_id,string,Cell identifier <aoi_id>_<row>_<col>,True
1,aoi_id,category,AOI identifier,True
2,year,int16,Year (start year for a season that crosses the...,False
3,season,category,wet / dry (AOI-specific months),False
4,B02_mean,float32,"B02 reflectance (0-1), mean over cell pixels",False
5,B03_mean,float32,"B03 reflectance (0-1), mean over cell pixels",False
6,B04_mean,float32,"B04 reflectance (0-1), mean over cell pixels",False
7,B05_mean,float32,"B05 reflectance (0-1), mean over cell pixels",False
8,B06_mean,float32,"B06 reflectance (0-1), mean over cell pixels",False
9,B07_mean,float32,"B07 reflectance (0-1), mean over cell pixels",False


## 3. Projected processing-unit cost (pilot pending)

**Why:** CDSE charges Process API requests by output size, number of input bands, number of data samples per pixel and output bit depth (Decision log #26). A multi-date composite passes *every* acquisition in the season to the evalscript, so it costs one "sample" per acquisition. These are **rule-based projections**; the real cost per request is measured in the pilot (M1-T05) and replaces these numbers.

Acquisitions per season are approximated as acquisition dates per year (from M1-T02's Catalog results) × season length / 12. Budget for the scaled extraction (M1-T06): 20,000 PU (ROADMAP §3.4).

In [4]:
avail = pd.read_csv(get_path(cfg, "results") / "m1_aoi_availability.csv")
dates_per_year = avail.groupby("aoi_id")["n_dates"].mean()
cap = cfg["quota"]["pu_caps"]["extraction"]
n_bands = len(cfg["sentinel2"]["bands"])

scenarios = {
    # name: (resolution m, input bands, float32 output)
    "A. 20 m, 10 bands + SCL, FLOAT32 (reference evalscript)": (20, n_bands + 1, True),
    "B. 20 m, 10 bands + SCL, UINT16 (reflectance x 10000)": (20, n_bands + 1, False),
    "C. 20 m, 6 index bands + SCL, UINT16 (B02 B03 B04 B08 B11 B12: all 9 indices; drops B05-B07, B8A)": (20, 7, False),
    "D. 100 m (cell size), 10 bands + SCL, UINT16 (no within-cell std)": (100, n_bands + 1, False),
}
proj = []
for name, (res, bands, f32) in scenarios.items():
    total_pu = 0.0
    for f in aois:
        p = f["properties"]
        w, h = grid_shape(p["bbox_utm"], res)
        for key in ("wet_season_months", "dry_season_months"):
            n_seasons = len(complete_seasons(p[key], study["start"], study["end"]))
            samples = dates_per_year[p["aoi_id"]] * season_fraction(p[key])
            total_pu += n_seasons * composite_pu(w, h, bands, samples, float32=f32)
    proj.append({"product": "seasonal composites (cell_season)", "scenario": name,
                 "requests": sum(len(complete_seasons(f["properties"][k], study["start"], study["end"]))
                                 for f in aois for k in ("wet_season_months", "dry_season_months")),
                 "projected_pu": round(total_pu), "share_of_extraction_cap": round(total_pu / cap, 2)})

# Monthly index series (cell_monthly): B02 B04 B08 B11 B12 + SCL in, 5 values out, one request per AOI-month.
for name, res, f32 in (("E. monthly indices, 20 m, FLOAT32", 20, True),
                        ("F. monthly indices, 100 m, FLOAT32", 100, True)):
    total_pu = 0.0
    for f in aois:
        w, h = grid_shape(f["properties"]["bbox_utm"], res)
        samples = dates_per_year[f["properties"]["aoi_id"]] / 12
        total_pu += n_months * composite_pu(w, h, 6, samples, float32=f32)
    proj.append({"product": "monthly indices (cell_monthly)", "scenario": name,
                 "requests": n_months * len(aois), "projected_pu": round(total_pu),
                 "share_of_extraction_cap": round(total_pu / cap, 2)})

projection = pd.DataFrame(proj)
projection.to_csv(RESULTS / "m1_pu_projection.csv", index=False)
projection

,product,scenario,requests,projected_pu,share_of_extraction_cap
0,seasonal composites (cell_season),"A. 20 m, 10 bands + SCL, FLOAT32 (reference ev...",81,92903,4.65
1,seasonal composites (cell_season),"B. 20 m, 10 bands + SCL, UINT16 (reflectance x...",81,46452,2.32
2,seasonal composites (cell_season),"C. 20 m, 6 index bands + SCL, UINT16 (B02 B03 ...",81,29560,1.48
3,seasonal composites (cell_season),"D. 100 m (cell size), 10 bands + SCL, UINT16 (...",81,1858,0.09
4,monthly indices (cell_monthly),"E. monthly indices, 20 m, FLOAT32",504,76904,3.85
5,monthly indices (cell_monthly),"F. monthly indices, 100 m, FLOAT32",504,3076,0.15


### Reading the projection

- The reference design (A) would cost several times the whole extraction budget: a full-resolution, 11-band, 32-bit composite over every acquisition in every season is too expensive for the free quota.
- Integer output (B) halves it; keeping only the six bands the nine indices need (C) cuts it further, and coarser output (D, F) cuts it most. Caveat for D/F: Sentinel Hub's default downsampling picks nearest pixels rather than averaging, so a 100 m request is not the same as a 100 m cell mean and loses the within-cell standard deviation. The choice of design is a scope decision for the project owner, made **after** the pilot (M1-T05) measures real costs.
- Whatever the design, the record and attribute counts above stay far above the Milestone 1 minimum: one AOI alone gives over 400,000 `cell_season` records.

## 4. Calibrated projection (after the pilot, M1-T05)

The pilot measured the real cost of two composites on a 5 km window (`reports/results/m1_pilot_cost.csv`, PU reported per request by the service). The ratio of measured to projected PU calibrates the rule-based projection, which is then recomputed for 16-bit designs only (Decision log #45) and for two further levers: smaller AOI squares, and passing only low-cloud acquisitions to the composite.

The low-cloud rows assume that Sentinel Hub's scene filter (maximum tile cloud cover) removes the filtered acquisitions from the charged samples; this is **not yet measured** and needs one cheap pilot request before it is relied on.

In [5]:
pilot_cost = pd.read_csv(RESULTS / "m1_pilot_cost.csv").set_index("request")
composites = [r for r in pilot_cost.index if r.startswith("composite")]
calib = pilot_cost.loc[composites, "reported_pu"].sum() / pilot_cost.loc[composites, "projected_pu"].sum()
print(f"Calibration factor (measured / projected PU): {calib:.3f}")

usable = avail.groupby("aoi_id")["n_usable_dates"].mean()
designs = cfg["pilot"]["designs"]

def projected_total(n_bands, size_m, low_cloud_only):
    pu = 0.0
    for f in aois:
        p = f["properties"]
        x0, y0, x1, y1 = p["bbox_utm"]
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        w, h = grid_shape([cx - size_m / 2, cy - size_m / 2, cx + size_m / 2, cy + size_m / 2], px_m)
        dates = (usable if low_cloud_only else dates_per_year)[p["aoi_id"]]
        for key in ("wet_season_months", "dry_season_months"):
            n = len(complete_seasons(p[key], study["start"], study["end"]))
            pu += n * composite_pu(w, h, n_bands + 1, dates * season_fraction(p[key]), float32=False)
    return pu * calib

rows = []
for design, bands in designs.items():
    for size_km in (20, 10):
        for low in (False, True):
            pu = projected_total(len(bands), size_km * 1000, low)
            rows.append({"design": design, "aoi_square_km": size_km,
                         "acquisitions": "low-cloud only (unmeasured)" if low else "all",
                         "projected_pu": round(pu), "share_of_extraction_cap": round(pu / cap, 2)})
calibrated = pd.DataFrame(rows)
calibrated.to_csv(RESULTS / "m1_pu_projection_calibrated.csv", index=False)
calibrated

Calibration factor (measured / projected PU): 1.020


,design,aoi_square_km,acquisitions,projected_pu,share_of_extraction_cap
0,b10,20,all,47395,2.37
1,b10,20,low-cloud only (unmeasured),32367,1.62
2,b10,10,all,11849,0.59
3,b10,10,low-cloud only (unmeasured),8092,0.40
4,c6,20,all,30160,1.51
5,c6,20,low-cloud only (unmeasured),20597,1.03
6,c6,10,all,7540,0.38
7,c6,10,low-cloud only (unmeasured),5149,0.26


## 5. Monthly index series (16-bit, 10 km squares)

The `cell_monthly` table needs one composite per AOI and month (84 months × 6 AOIs = 504 requests). It takes the five bands the trend indices need (B02 B04 B08 B11 B12 for NDVI, MSAVI2, BSI and albedo) plus SCL, and outputs 16-bit median reflectances and a valid count; the indices are computed locally. Costs use the pilot calibration and 10 km squares, at 20 m and at 100 m (one pixel per cell; Sentinel Hub picks rather than averages pixels when downsampling).

The plan totals add the seasonal composites (10 km squares) to each monthly option and compare with the 20 000 PU extraction budget.

In [6]:
monthly_bands = cfg["pilot"]["monthly_bands"]
size_m = 10_000

def monthly_total(res, low_cloud_only):
    pu = 0.0
    for f in aois:
        p = f["properties"]
        x0, y0, x1, y1 = p["bbox_utm"]
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        w, h = grid_shape([cx - size_m / 2, cy - size_m / 2, cx + size_m / 2, cy + size_m / 2], res)
        dates = (usable if low_cloud_only else dates_per_year)[p["aoi_id"]]
        pu += n_months * composite_pu(w, h, len(monthly_bands) + 1, dates / 12, float32=False)
    return pu * calib

monthly = pd.DataFrame([
    {"product": "monthly series", "resolution_m": res,
     "acquisitions": "low-cloud only (unmeasured)" if low else "all",
     "requests": n_months * len(aois), "projected_pu": round(monthly_total(res, low))}
    for res in (20, 100) for low in (False, True)
])
monthly.to_csv(RESULTS / "m1_pu_projection_monthly.csv", index=False)

seasonal_10km = calibrated[calibrated["aoi_square_km"] == 10]
plans = []
for _, s_row in seasonal_10km.iterrows():
    for _, m_row in monthly.iterrows():
        if s_row["acquisitions"] != m_row["acquisitions"]:
            continue  # pair like with like: both all, or both low-cloud
        total = s_row["projected_pu"] + m_row["projected_pu"]
        plans.append({"seasonal_design": s_row["design"], "monthly_resolution_m": m_row["resolution_m"],
                      "acquisitions": s_row["acquisitions"], "seasonal_pu": s_row["projected_pu"],
                      "monthly_pu": m_row["projected_pu"], "total_pu": total,
                      "share_of_extraction_cap": round(total / cap, 2)})
plans = pd.DataFrame(plans).sort_values("total_pu").reset_index(drop=True)
plans.to_csv(RESULTS / "m1_pu_plan_options.csv", index=False)
display(monthly)
plans

,product,resolution_m,acquisitions,requests,projected_pu
0,monthly series,20,all,504,9808
1,monthly series,20,low-cloud only (unmeasured),504,6702
2,monthly series,100,all,504,392
3,monthly series,100,low-cloud only (unmeasured),504,268


,seasonal_design,monthly_resolution_m,acquisitions,seasonal_pu,monthly_pu,total_pu,share_of_extraction_cap
0,c6,100,low-cloud only (unmeasured),5149,268,5417,0.27
1,c6,100,all,7540,392,7932,0.40
2,b10,100,low-cloud only (unmeasured),8092,268,8360,0.42
3,c6,20,low-cloud only (unmeasured),5149,6702,11851,0.59
4,b10,100,all,11849,392,12241,0.61
5,b10,20,low-cloud only (unmeasured),8092,6702,14794,0.74
6,c6,20,all,7540,9808,17348,0.87
7,b10,20,all,11849,9808,21657,1.08
